# 🚀 Leocherd Iris Energy — GPU 3D Motion & Rigging Reconstruction

Welcome to the official **Leocherd Iris Energy** cloud GPU acceleration pipeline.
This notebook runs on **free Kaggle GPUs (Tesla T4 / P100)** to perform high-fidelity, high-fps 3D human pose tracking, biomechanical regularization, and temporal motion smoothing on any video using the modern **MediaPipe 1.x Tasks API**.

### Workflow Overview:
1. **Upload Video**: Upload any `.mp4` or `.mov` clip to Kaggle (or attach as Dataset).
2. **GPU Inference**: Google MediaPipe Heavy Task model runs on CUDA GPU.
3. **Temporal Filtering**: Savitzky-Golay smoothing eliminates monocular jitter and Z-axis popping.
4. **Kinematic Calibration**: World coordinates are transformed into canonical Leocherd Iris skeleton hierarchy.
5. **Export & Download**: Produces `iris_kaggle_pose_data.json` ready for local 1-click 3D character rigging!


In [ ]:
# 1. Environment & Hardware Verification
!nvidia-smi

!pip install -q mediapipe opencv-python-headless scipy plotly
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task

import os
import glob
import json
import cv2
import numpy as np
import torch
from scipy.signal import savgol_filter
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ PyTorch running on: {device}')
if torch.cuda.is_available():
    print(f'   Device Name: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Detect Uploaded Video
videos = glob.glob('/kaggle/input/**/*.mp4', recursive=True) + glob.glob('/kaggle/working/*.mp4')
if not videos:
    raise RuntimeError('⚠️ No MP4 video found! Please upload a video or attach your dataset.')
INPUT_VIDEO = videos[0]
print(f'📹 Found video: {INPUT_VIDEO}')


In [ ]:
# 3. Initialize Modern MediaPipe 1.x Heavy Pose Model with GPU Delegate
if torch.cuda.is_available():
    print(f'🚀 GPU DETECTED: {torch.cuda.get_device_name(0)}')
    print('⚡ Activating MediaPipe BaseOptions.Delegate.GPU for maximum speed!')
    delegate = mp_tasks.BaseOptions.Delegate.GPU
else:
    print('⚠️ NO GPU DETECTED: Kaggle Accelerator is currently set to "None".')
    print('👉 To enable GPU: in the Kaggle top menu, click Settings -> Accelerator -> GPU T4 x2')
    print('ℹ️ Running on CPU for now...')
    delegate = mp_tasks.BaseOptions.Delegate.CPU

base_options = mp_tasks.BaseOptions(
    model_asset_path='pose_landmarker.task',
    delegate=delegate
)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
detector = vision.PoseLandmarker.create_from_options(options)
print(f'✅ MediaPipe 1.x Pose Landmarker initialized with delegate: {delegate.name}')

LANDMARK_MAP = {
    0: 'Nose',
    7: 'LeftEar', 8: 'RightEar',
    11: 'LeftShoulder', 12: 'RightShoulder',
    13: 'LeftArm', 14: 'RightArm',
    15: 'LeftForeArm', 16: 'RightForeArm',
    17: 'LeftHand', 18: 'RightHand',
    23: 'LeftUpLeg', 24: 'RightUpLeg',
    25: 'LeftLeg', 26: 'RightLeg',
    27: 'LeftFoot', 28: 'RightFoot',
}


In [ ]:
# 4. Process Video & Extract Metric World Landmarks
cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Processing {total_frames} frames @ {fps:.2f} FPS...')

raw_frames = []
frame_idx = 0

while cap.isOpened() and frame_idx < 600:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    frame_joints = {}
    if result.pose_world_landmarks:
        wl = result.pose_world_landmarks[0]
        l_hip, r_hip = wl[23], wl[24]
        l_sh, r_sh = wl[11], wl[12]
        nose = wl[0]
        l_ear, r_ear = wl[7], wl[8]

        # Hips (Pelvis Center)
        hips_x = (l_hip.x + r_hip.x) * 0.5
        hips_y = -(l_hip.y + r_hip.y) * 0.5 + 1.0
        hips_z = (l_hip.z + r_hip.z) * 0.5
        frame_joints['Hips'] = {'x': float(hips_x), 'y': float(hips_y), 'z': float(hips_z), 'confidence': 1.0}

        # Chest (Shoulder Center)
        chest_x = (l_sh.x + r_sh.x) * 0.5
        chest_y = -(l_sh.y + r_sh.y) * 0.5 + 1.0
        chest_z = (l_sh.z + r_sh.z) * 0.5
        frame_joints['Chest'] = {'x': float(chest_x), 'y': float(chest_y), 'z': float(chest_z), 'confidence': 1.0}

        # Spine (Spine Center)
        frame_joints['Spine'] = {
            'x': float((hips_x + chest_x) * 0.5),
            'y': float((hips_y + chest_y) * 0.5),
            'z': float((hips_z + chest_z) * 0.5),
            'confidence': 1.0
        }

        # Head (Ear Midpoint with cranial offset)
        if (l_ear.visibility or 0) > 0.4 and (r_ear.visibility or 0) > 0.4:
            head_x = (l_ear.x + r_ear.x) * 0.5
            head_y = -(l_ear.y + r_ear.y) * 0.5 + 1.0
            head_z = (l_ear.z + r_ear.z) * 0.5
        else:
            head_x = nose.x
            head_y = -nose.y + 1.0
            head_z = nose.z - 0.05

        frame_joints['Head'] = {
            'x': float(head_x),
            'y': float(head_y),
            'z': float(head_z),
            'confidence': float(nose.visibility or 1.0)
        }

        # Neck (Dynamic anatomical bridge smoothly following Head and Spine)
        frame_joints['Neck'] = {
            'x': float(chest_x * 0.45 + head_x * 0.55),
            'y': float(chest_y * 0.45 + head_y * 0.55),
            'z': float(chest_z * 0.45 + head_z * 0.55),
            'confidence': 1.0
        }

        for lm_idx, j_name in LANDMARK_MAP.items():
            lm = wl[lm_idx]
            frame_joints[j_name] = {
                'x': float(lm.x),
                'y': float(-lm.y + 1.0),
                'z': float(lm.z),
                'confidence': float(lm.visibility or 1.0),
                'visibility': float(lm.visibility or 1.0)
            }

    raw_frames.append({
        'frame_index': frame_idx,
        'timestamp_sec': float(frame_idx) / float(fps),
        'joints': frame_joints
    })
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f'   Processed {frame_idx}/{total_frames} frames...')

cap.release()
print(f'✅ Extraction finished: {len(raw_frames)} frames.')


In [ ]:
# 5. Apply Temporal Savitzky-Golay Smoothing
if len(raw_frames) >= 7:
    all_joint_names = set()
    for f in raw_frames:
        all_joint_names.update(f['joints'].keys())

    T = len(raw_frames)
    for j_name in all_joint_names:
        traj = np.zeros((T, 3), dtype=np.float32)
        for t, f in enumerate(raw_frames):
            j = f['joints'].get(j_name, {'x': 0.0, 'y': 0.0, 'z': 0.0})
            traj[t, 0] = j['x']
            traj[t, 1] = j['y']
            traj[t, 2] = j['z']

        for d in range(3):
            traj[:, d] = savgol_filter(traj[:, d], window_length=7, polyorder=2)

        for t, f in enumerate(raw_frames):
            if j_name in f['joints']:
                f['joints'][j_name]['x'] = float(traj[t, 0])
                f['joints'][j_name]['y'] = float(traj[t, 1])
                f['joints'][j_name]['z'] = float(traj[t, 2])
    print('✅ Savitzky-Golay motion regularisation complete.')


In [ ]:
# 6. Export to Iris JSON
out_path = '/kaggle/working/iris_kaggle_pose_data.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump({
        'generator': 'Leocherd Iris Energy MediaPipe 1.x GPU Engine 0.2.0',
        'morphology': 'human',
        'fps': float(fps),
        'total_frames': len(raw_frames),
        'frames': raw_frames,
    }, f, indent=2)

print(f'🎉 SUCCESS! Saved 3D motion data to: {out_path}')
print('👉 In the right panel under Output (/kaggle/working), click ... next to iris_kaggle_pose_data.json and choose Download!')
